# Person mentions by word position with an LLM on the DHinfra cluster

Ten letters (German, English, Italian) from the TEI edition of the Hammer-Purgstall
correspondence (1794-1856). The edition marks persons only in part: often only the surname is a
`<persName>`, some names are split over two elements, some `<persName>` are not persons at all,
and pronouns ("er", "Sie", "mio amico") are not marked. The task: mark every mention of a real
person, with its name parts (`roleName`, `forename`, `nameLink`, `surname` ...), and every pronoun
or phrase that refers to one as `<rs>`.

**The model never writes the letter.** It gets the letter text numbered word by word and answers
with word numbers only. The script sets the tags, so the letter text cannot change. Notebook 2
does the same task the other way round: the model gets the TEI and writes it back annotated.

There is no reference annotation in this example. The checks in section 5 say whether an answer
is usable and what it changed in the edition's markup, not whether it is right. No GPU is needed
in your own session: the model runs on the cluster.

**Steps:** 1. set up, 2. from TEI to numbered text, 3. one letter, 4. all ten letters,
5. check the results.

## 1. Set up

In [ ]:
# Run once per session. The cluster's Python environment may already have these.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

Your project token is read from `DHINFRA_KEY`, either from the environment or from a `.env`
file next to this notebook (`DHINFRA_KEY="..."`). Never commit the `.env` file.

In [ ]:
import json
from pathlib import Path
import concurrent.futures as cf

import pandas as pd

import helpers

MODEL = "deepseek-v4.1-flash"   # pick a name from the list printed below
THINKING = False                # True: the model reasons first (slower, more tokens)
WORKERS = 4                     # parallel requests; keep this small, the cluster is shared

PROMPT_DIR = Path("prompts/positions")
LETTER_DIR = Path("data/letters")
METADATA_FILE = Path("data/metadata.json")
OUTPUT_DIR = Path("output") / MODEL / "positions"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = helpers.get_client()
# Optional: list the models the API offers, to check that MODEL is spelled correctly.
print([m.id for m in client.models.list().data])

## 2. From TEI to numbered text

`data/letters/` holds the letter body of each letter as it is in the edition
(`<div type="letter">`). The model does not get this XML but its plain text:

- the footnotes are taken out (they are not annotated) and listed separately, as context;
- the end of a block (`<p>`, `<salute>`, `<signed>` ...) ends a word;
- whitespace is collapsed to single spaces.

Then the words are numbered. A word is everything between two spaces, so punctuation stays with
its word ("Copenhagen," is one word).

`letter2276` is a short Italian letter. The edition marks "Muenter" twice and "Morelli" once.

In [ ]:
metadata = json.loads(METADATA_FILE.read_text(encoding="utf-8"))

name = "letter2276"
letter = helpers.letter_text(helpers.read_letter(LETTER_DIR / f"{name}.xml"))
print(letter["text"])
print()
print(len(letter["tokens"]), "words,", len(letter["notes"]), "footnotes")
print("existing markup:", [(s["element"], s["text"]) for s in helpers.existing_markup(letter)])

The *user message* has four blocks: who wrote the letter to whom (with the IDs of the edition's
person index), the numbered text, the markup the edition already has, and the footnotes.

In [ ]:
print(helpers.positions_message(metadata[name], letter))

Three files make up the *system prompt*: `prompt.txt` (the task, in five steps),
`encoding_rules.txt` (what belongs into a `persName`, what is an `rs`) and
`few_shot_examples.txt` (the answer format and one worked example).

In [ ]:
system_prompt = helpers.build_system_prompt(PROMPT_DIR)
print(f"{len(system_prompt):,} characters")
print(system_prompt[:2500], "...")

## 3. One letter

The answer is a JSON list of mentions, each with its first and last word number:
`{"element": "persName", "tok_start": 4, "tok_end": 5, "parts": [...]}`.

In [ ]:
raw, usage = helpers.ask_model(
    client, MODEL, system_prompt, helpers.positions_message(metadata[name], letter), thinking=THINKING)
print(raw)
print()
print("tokens in/out:", usage.prompt_tokens, usage.completion_tokens)

The script turns the word numbers into character positions in the letter text. On the way it
removes punctuation that cannot belong to a name ("Copenhagen," -> "Copenhagen"), cuts a
grammatical ending the model marked as `suffix` ("Wielands" -> "Wieland"), and rejects what the
TEI could not hold: a name part outside its name, two mentions that overlap, a mention that runs
from one text block into the next (from the salutation into the first paragraph, say). Each
correction is listed under *issues*.

In [ ]:
spans, issues = helpers.positions_to_spans(helpers.parse_json_answer(raw), letter)
for s in spans:
    parts = ", ".join(f"{p['element']}={p['text']!r}" for p in s["parts"])
    print(f"{s['element']:9} {s['text']!r:40} {parts}")
print()
print("issues:", issues or "none")

Last, the tags go into the text. The text between the tags is copied from the letter, character
by character, so the check below compares the plain text of the result with the letter text.

In [ ]:
xml = helpers.spans_to_tei(letter["text"], spans)
print(xml)
print()
print("letter text unchanged:", helpers.text_of(xml) == letter["text"])

## 4. All ten letters

One request per letter, a few at a time. For each letter, `output/<model>/positions/` gets a
`.json` file (the mentions as character positions, the issues, the token counts) and an `.xml`
file (the annotated text).

In [ ]:
def run_one(name):
    letter = helpers.letter_text(helpers.read_letter(LETTER_DIR / f"{name}.xml"))
    raw, usage = helpers.ask_model(
        client, MODEL, system_prompt, helpers.positions_message(metadata[name], letter),
        thinking=THINKING)
    answer = helpers.parse_json_answer(raw)
    spans, issues = helpers.positions_to_spans(answer, letter)
    xml = helpers.spans_to_tei(letter["text"], spans)
    result = {"valid_json": answer is not None, "spans": spans, "issues": issues,
              "tokens_in": usage.prompt_tokens, "tokens_out": usage.completion_tokens}
    (OUTPUT_DIR / f"{name}.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    (OUTPUT_DIR / f"{name}.xml").write_text(xml, encoding="utf-8")
    return letter, result, xml

results = {}
with cf.ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(run_one, n): n for n in metadata}
    for future in cf.as_completed(futures):
        n = futures[future]
        try:
            results[n] = future.result()
            print("done  ", n)
        except Exception as err:   # keep going; one failed letter should not stop the batch
            print("FAILED", n, "-", err)

## 5. Check the results

- **valid_json**: the answer could be read.
- **persName / rs**: how many mentions the model marked.
- **issues**: how many corrections the script had to make (see the list per letter below).
- **text_unchanged**: the plain text of the result is the letter text. By construction it always
  is; notebook 2 has no such guarantee.
- **kept ... added**: what happened to the edition's `persName` markup. *kept*: same span.
  *extended*: the model widened or narrowed it ("Muenter" -> "Federico Muenter"). *merged*: one
  name made of several old spans. *removed*: an old `persName` the model did not keep (a work
  title, a periodical, or a mistake). *added*: a name the edition did not mark.

Whether the right words were marked needs a look at the XML.

In [ ]:
rows = []
for n, (letter, result, xml) in sorted(results.items()):
    rows.append({
        "letter": n,
        "language": metadata[n]["language"],
        "valid_json": result["valid_json"],
        "persName": sum(s["element"] == "persName" for s in result["spans"]),
        "rs": sum(s["element"] == "rs" for s in result["spans"]),
        "issues": len(result["issues"]),
        "text_unchanged": helpers.text_of(xml) == letter["text"],
        **helpers.compare_with_edition(letter, result["spans"]),
        "tokens_out": result["tokens_out"],
    })
table = pd.DataFrame(rows).set_index("letter")
table

In [ ]:
# One letter in detail. Change the name to see another one.
n = "letter1222"
letter, result, xml = results[n]
print("issues:", *result["issues"] or ["none"], sep="\n  ")
print()
print(xml)

## Where next

- Change `MODEL`, set `THINKING = True`, or change the rules in `prompts/positions/`, and run
  again. The output goes to a folder named after the model.
- Run `02_persons_in_tei.ipynb` with the same `MODEL`: it does the same task on the TEI and
  compares its result with this one.
- The full pipeline writes the tags back into the TEI file at the places they came from, and then
  links each mention to the person index or the GND. Both steps are not part of this example.